In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from PIL import Image
import os

# Configurações OTIMIZADAS
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "runwayml/stable-diffusion-v1-5"

# Verificar se temos GPU para usar float16
if device == "cuda":
    torch_dtype = torch.float16  # Usar half precision na GPU
    variant = "fp16"
else:
    torch_dtype = torch.float32  # Usar float32 na CPU
    variant = None

print(f"Usando dispositivo: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")

# Carregar o modelo COM CONFIGURAÇÃO CORRETA
pipe = StableDiffusionPipeline.from_pretrained(
    model_id, 
    torch_dtype=torch_dtype,
    variant=variant,
    safety_checker=None,
    requires_safety_checker=False
)

# Configurar agendador mais rápido
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

# Apenas habilitar attention slicing se estiver na GPU com pouca VRAM
if device == "cuda" and torch.cuda.get_device_properties(0).total_memory < 10e9:  # < 10GB VRAM
    pipe.enable_attention_slicing()

# Prompts definidos
prompts = [
    "A beautiful sunset over mountains, digital art",
    "A futuristic city with flying cars, cyberpunk style",
    "A cute cat wearing a hat, cartoon style",
    "An ancient temple in the jungle, photorealistic"
]

# Hiperparâmetros OTIMIZADOS - AJUSTADOS para CPU/GPU
num_images_per_prompt = 3  # REDUZIDO para 3 (suficiente para análise)
guidance_scale = 7.5

# Ajustar steps baseado no dispositivo
if device == "cuda":
    num_inference_steps = 20  # GPU pode ser mais rápido
else:
    num_inference_steps = 15  # CPU precisa de menos steps

print(f"Configuração: {num_inference_steps} steps, {num_images_per_prompt} imagens por prompt")

# Gerar imagens
for i, prompt in enumerate(prompts):
    print(f"\nGerando imagens para: {prompt}")
    os.makedirs(f"generated_images/prompt_{i}", exist_ok=True)
    
    try:
        # Gera as imagens
        images = pipe(
            prompt=prompt,
            guidance_scale=guidance_scale,
            num_inference_steps=num_inference_steps,
            num_images_per_prompt=num_images_per_prompt
        ).images
        
        # Salva as imagens
        for j, image in enumerate(images):
            image.save(f"generated_images/prompt_{i}/image_{j}.png")
            print(f"✓ Imagem {j+1} salva")
            
    except Exception as e:
        print(f"Erro ao gerar imagens: {e}")
        # Fallback: gerar uma imagem por vez se batch falhar
        for j in range(num_images_per_prompt):
            try:
                image = pipe(
                    prompt=prompt,
                    guidance_scale=guidance_scale,
                    num_inference_steps=num_inference_steps
                ).images[0]
                image.save(f"generated_images/prompt_{i}/image_{j}.png")
                print(f"✓ Imagem {j+1} salva (fallback)")
            except Exception as e2:
                print(f"Erro na geração individual: {e2}")

print("\n✅ Geração concluída!")

Geração de imagens Otimizada

In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from PIL import Image
import os

# Configurações OTIMIZADAS
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "runwayml/stable-diffusion-v1-5"

# Carregar o modelo COM OTIMIZAÇÕES
pipe = StableDiffusionPipeline.from_pretrained(
    model_id, 
    torch_dtype=torch.float16,  # Half precision - REDUZ MEMÓRIA pela METADE
    variant="fp16",             # Garante compatibilidade com float16
    safety_checker=None,        # Remove verificador de segurança (mais rápido)
    requires_safety_checker=False
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)  # Agendador mais rápido
pipe = pipe.to(device)
pipe.enable_attention_slicing()  # Reduz uso de memória em GPUs com pouca VRAM

# Prompts definidos (mantidos)
prompts = [
    "A beautiful sunset over mountains, digital art",
    "A futuristic city with flying cars, cyberpunk style",
    "A cute cat wearing a hat, cartoon style",
    "An ancient temple in the jungle, photorealistic"
]

# Hiperparâmetros OTIMIZADOS
num_images_per_prompt = 2  # REDUZIDO: 5 em vez de 10 (ainda é suficiente)
guidance_scale = 7.5
num_inference_steps = 15   # REDUZIDO PELA METADE: 25 em vez de 50

# Gerar imagens de forma OTIMIZADA
for i, prompt in enumerate(prompts):
    print(f"Gerando imagens para: {prompt}")
    os.makedirs(f"generated_images/prompt_{i}", exist_ok=True)
    
    # Gera todas as imagens de uma vez (BATCH processing)
    images = pipe(
        prompt=prompt,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        num_images_per_prompt=num_images_per_prompt  # Gera várias de uma vez
    ).images
    
    # Salva todas as imagens do batch
    for j, image in enumerate(images):
        image.save(f"generated_images/prompt_{i}/image_{j}.png")

print("Geração concluída!")